# 🧠 Emora Text Emotion Model — v3
### XLM-RoBERTa Multilingual — Arabic (Egyptian) + English
**Labels:** `angry | fear | happy | neutral | sad | surprise`

## 1️⃣ Install & Import

In [ ]:
!pip install transformers datasets scikit-learn -q
import pandas as pd, numpy as np, os, re, torch
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
print('✅ Libraries loaded!')
print(f'🔥 GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT AVAILABLE"}')

## 2️⃣ Load Data

In [ ]:
BASE  = '/kaggle/input/datasets/rawanelmorsy/emotions/USEFUL/'
BASE2 = '/kaggle/input/datasets/rawanelmorsy/arabic/'

FILES = {
    'emotion1': BASE  + 'emotion1.csv',
    'emotion3': BASE  + 'emotion3.csv',
    'emotion4': BASE  + 'emotion4.csv',
    'emotion5': BASE  + 'emotion5.csv',
    'emotion6': BASE  + 'emotion6.csv',
    'arabic':   BASE2 + 'arabic_full_dataset.csv',  # ✅ الداتا العربي الجديد
}

raw = {}
for name, path in FILES.items():
    df = pd.read_csv(path)
    raw[name] = df
    print(f'✅ {name}: {df.shape}')
print('\n⚠️ emotion2 skipped (identical to emotion6)')

## 3️⃣ EDA

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for idx, (name, df) in enumerate(raw.items()):
    if idx >= len(axes): break
    counts = df['label'].value_counts()
    axes[idx].bar(counts.index, counts.values, color='steelblue', edgecolor='white')
    axes[idx].set_title(f'{name} — {len(df):,} rows', fontsize=11, fontweight='bold')
    axes[idx].tick_params(axis='x', rotation=45)
plt.suptitle('📊 Raw Label Distribution per File', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

sample = pd.concat([df.sample(min(3000, len(df)), random_state=42) for df in raw.values()])
sample['text_len'] = sample['text'].astype(str).apply(lambda x: len(x.split()))
plt.figure(figsize=(10, 4))
plt.hist(sample['text_len'], bins=50, color='steelblue', edgecolor='white')
plt.axvline(sample['text_len'].median(), color='red', linestyle='--', label=f'Median: {sample["text_len"].median():.0f}')
plt.axvline(sample['text_len'].quantile(0.95), color='orange', linestyle='--', label=f'95th: {sample["text_len"].quantile(0.95):.0f}')
plt.title('📏 Text Length Distribution', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

## 4️⃣ Preprocessing & Label Mapping

In [ ]:
LABEL_MAP = {
    'joy':'happy','happiness':'happy','fun':'happy','enthusiasm':'happy',
    'relief':'happy','love':'happy','admiration':'happy','amusement':'happy',
    'approval':'happy','caring':'happy','excitement':'happy','gratitude':'happy',
    'optimism':'happy','pride':'happy',
    'sad':'sad','sadness':'sad','empty':'sad','grief':'sad',
    'remorse':'sad','disappointment':'sad','embarrassment':'sad',
    'anger':'angry','hate':'angry','annoyance':'angry','disapproval':'angry','disgust':'angry',
    'fear':'fear','worry':'fear','nervousness':'fear',
    'surprise':'surprise','realization':'surprise','confusion':'surprise','curiosity':'surprise',
    'neutral':'neutral','boredom':'neutral','desire':'neutral',
    # العربي (من emotion1_arabic الأصلي)
    'happy':'happy','sad':'sad','angry':'angry','fear':'fear','neutral':'neutral','surprise':'surprise',
}

def clean_text(text):
    text = str(text).strip()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+|#\w+', '', text)
    text = re.sub(r'[^\w\s\u0600-\u06FF!?.,\'\-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

dfs = []
for name, df in raw.items():
    df = df.copy()
    df['text'] = df['text'].apply(clean_text)

    # ✅ الداتا العربي label جاهز — مش محتاج mapping
    if name == 'arabic':
        df = df[['text','label']]
        df = df[df['text'].apply(lambda x: len(x.split()) >= 1)]
        dfs.append(df)
        print(f'✅ {name}: {len(df)} rows (Arabic — no mapping needed)')
        continue

    before = len(df)
    df['emotion'] = df['label'].map(LABEL_MAP)
    df = df.dropna(subset=['emotion'])
    df = df[df['text'].apply(lambda x: len(x.split()) >= 2)]
    df = df.drop_duplicates(subset=['text'])
    df = df[['text','emotion']].rename(columns={'emotion':'label'})
    dfs.append(df)
    print(f'✅ {name}: {before:,} → {len(df):,}')

combined = pd.concat(dfs, ignore_index=True)
before_global = len(combined)
combined = combined.drop_duplicates(subset=['text'])
print(f'\n🔁 Global dedup: {before_global:,} → {len(combined):,}')
print('\n📊 Final distribution:')
print(combined['label'].value_counts())

## 5️⃣ Balance Dataset

In [ ]:
MAX_PER_LABEL = 10000
balanced = []
for label in combined['label'].unique():
    subset = combined[combined['label'] == label]
    balanced.append(subset.sample(min(MAX_PER_LABEL, len(subset)), random_state=42))

df_balanced = pd.concat(balanced, ignore_index=True).sample(frac=1, random_state=42)
print('📊 After balancing:')
print(df_balanced['label'].value_counts())
print(f'\nTotal: {len(df_balanced):,} rows')

plt.figure(figsize=(10, 4))
counts = df_balanced['label'].value_counts()
colors = ['#e74c3c','#3498db','#2ecc71','#95a5a6','#f39c12','#1abc9c']
bars = plt.bar(counts.index, counts.values, color=colors, edgecolor='white')
plt.title('✅ Balanced Label Distribution (6 classes)', fontsize=13, fontweight='bold')
for bar, v in zip(bars, counts.values):
    plt.text(bar.get_x()+bar.get_width()/2, v+50, f'{v:,}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

## 6️⃣ Train / Val Split

In [ ]:
CLASSES  = ['angry','fear','happy','neutral','sad','surprise']
label2id = {l:i for i,l in enumerate(CLASSES)}
id2label = {i:l for i,l in enumerate(CLASSES)}
df_balanced['label_id'] = df_balanced['label'].map(label2id)

train_df, val_df = train_test_split(
    df_balanced, test_size=0.15,
    stratify=df_balanced['label_id'], random_state=42
)
print(f'✅ Train: {len(train_df):,} | Val: {len(val_df):,}')
print(train_df['label'].value_counts())

## 7️⃣ Tokenizer & Dataset

In [ ]:
MODEL_NAME = 'xlm-roberta-base'
MAX_LEN    = 64
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

class EmotionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts=texts.reset_index(drop=True); self.labels=labels.reset_index(drop=True)
        self.tok=tokenizer; self.max_len=max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tok(str(self.texts[idx]), max_length=self.max_len,
                       padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids':enc['input_ids'].squeeze(),
                'attention_mask':enc['attention_mask'].squeeze(),
                'labels':torch.tensor(self.labels[idx], dtype=torch.long)}

train_dataset = EmotionDataset(train_df['text'], train_df['label_id'], tokenizer, MAX_LEN)
val_dataset   = EmotionDataset(val_df['text'],   val_df['label_id'],   tokenizer, MAX_LEN)
print(f'✅ Train: {len(train_dataset):,} | Val: {len(val_dataset):,}')

## 8️⃣ Train

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(CLASSES), id2label=id2label, label2id=label2id)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return {'accuracy': accuracy_score(labels, np.argmax(logits, axis=-1))}

training_args = TrainingArguments(
    output_dir='/kaggle/working/emora_text_ckpt',
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=200,
    weight_decay=0.01,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    report_to='none',
)

trainer = Trainer(model=model, args=training_args,
                  train_dataset=train_dataset, eval_dataset=val_dataset,
                  compute_metrics=compute_metrics)
print('🚀 Training started...')
trainer.train()

## 9️⃣ Evaluate

In [ ]:
results = trainer.evaluate()
print(f'🎯 Final Validation Accuracy: {results["eval_accuracy"]*100:.2f}%')

preds_out = trainer.predict(val_dataset)
preds = np.argmax(preds_out.predictions, axis=-1)
print('\n📊 Classification Report:')
print(classification_report(val_df['label_id'], preds, target_names=CLASSES))

cm = confusion_matrix(val_df['label_id'], preds)
plt.figure(figsize=(9,7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES)
plt.title('🔍 Confusion Matrix', fontsize=13, fontweight='bold')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.tight_layout(); plt.show()

## 🔟 Save Model

In [ ]:
SAVE_PATH = '/kaggle/working/emora_text_model_final'
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f'✅ Saved to: {SAVE_PATH}')
for f in os.listdir(SAVE_PATH):
    size = os.path.getsize(os.path.join(SAVE_PATH,f))/(1024*1024)
    print(f'  {f}  ({size:.1f} MB)')

## 1️⃣1️⃣ Quick Test — Arabic & English

In [ ]:
def predict_emotion(text, threshold=0.4):
    inputs = tokenizer(text, return_tensors='pt', truncation=True,
                       max_length=MAX_LEN, padding=True).to(model.device)
    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(**inputs).logits, dim=-1)[0]
    conf = probs.max().item()
    pred = id2label[probs.argmax().item()]
    # طباعة كل الاحتمالات للتشخيص
    all_probs = {id2label[i]: round(float(probs[i])*100,2) for i in range(len(CLASSES))}
    return ('unknown' if conf < threshold else pred), round(conf*100,2), all_probs

tests = [
    ('I am so happy today!',            'happy'),
    ('أنا زعلان جداً',                  'sad'),
    ('I feel scared and nervous',       'fear'),
    ('أنا فرحان ومبسوط',                'happy'),
    ('I am very angry right now',       'angry'),
    ('خايف من الظلام',                  'fear'),
    ('What a surprise!',                'surprise'),
    ('أنا مش حاسس بحاجة',              'neutral'),
    ('I hate this so much',             'angry'),
    ('الأكل ده ريحته وحشة جداً',        'angry'),
    ('I feel so sad and lonely today',  'sad'),
    ('مش متوقعت ده خالص',              'surprise'),
]

emoji_map = {'happy':'😄','sad':'😢','angry':'😠','fear':'😨',
             'surprise':'😲','neutral':'😐','unknown':'🤔'}
print('🧪 Test Results:')
print('='*60)
correct = 0
for text, expected in tests:
    emotion, conf, all_p = predict_emotion(text)
    status = '✅' if emotion == expected else '❌'
    print(f'  {status} {text}')
    print(f'     → {emoji_map.get(emotion,"")} {emotion.upper()} ({conf}%)  [expected: {expected}]')
    if emotion != expected:
        print(f'     all scores: {all_p}')
    print()
    if emotion == expected: correct += 1
print(f'Quick Test Score: {correct}/{len(tests)} ({correct/len(tests)*100:.0f}%)')